# Sumativa 4 — Taller 3
## Integración, limpieza y análisis de ventas con Apache Spark

**Curso:** Gestión de Datos y Tecnologías Big Data — MCDI502  
**Grupo:** 7  
**Entorno:** Databricks

### Objetivo

Integrar las ventas del último trimestre de las sucursales de Viña del Mar, Santiago y Concepción. El flujo revisa la calidad de los datos, aplica las reglas de limpieza solicitadas, crea variables para el análisis y ejecuta consultas con Spark SQL.

El notebook está organizado en las cinco evidencias principales de la rúbrica: carga e integración, limpieza, transformación, validación de calidad y consultas SQL.

# PARTE 1: Configuración inicial y carga de archivos (ID3.1)

Los CSV deben cargarse previamente en DBFS. En Databricks se puede usar **Add data → Upload files to DBFS**. Las rutas se concentran en una sola celda para que puedan modificarse sin cambiar el resto del notebook.

Databricks proporciona la variable spark; por eso no se instala PySpark ni se crea otra sesión.

In [ ]:
from pyspark.sql import functions as F

print("Versión de Spark:", spark.version)

# Rutas de los tres archivos cargados en DBFS.
RUTA_VINA = "dbfs:/FileStore/tables/ventas_vina_del_mar.csv"
RUTA_SANTIAGO = "dbfs:/FileStore/tables/ventas_santiago.csv"
RUTA_CONCEPCION = "dbfs:/FileStore/tables/ventas_concepcion.csv"

rutas = [RUTA_VINA, RUTA_SANTIAGO, RUTA_CONCEPCION]
for ruta in rutas:
    try:
        dbutils.fs.ls(ruta)
        print("Archivo disponible:", ruta)
    except Exception as error:
        raise FileNotFoundError(
            f"No se encontró {ruta}. Carga el CSV en DBFS y vuelve a ejecutar esta celda."
        ) from error

## Carga de los tres DataFrames

Se leen inicialmente como texto para evitar que los formatos mezclados de fecha y precio se conviertan de manera incorrecta. Las conversiones se realizan de forma explícita en la etapa de limpieza.

In [ ]:
opciones_csv = {"header": True, "inferSchema": False}

df_vina = spark.read.options(**opciones_csv).csv(RUTA_VINA)
df_santiago = spark.read.options(**opciones_csv).csv(RUTA_SANTIAGO)
df_concepcion = spark.read.options(**opciones_csv).csv(RUTA_CONCEPCION)

def inspeccionar(nombre, df):
    print(f"\n--- {nombre} ---")
    print("Filas:", df.count())
    df.printSchema()
    df.show(5, truncate=False)

inspeccionar("Viña del Mar", df_vina)
inspeccionar("Santiago", df_santiago)
inspeccionar("Concepción", df_concepcion)

## Unión de los DataFrames

Los tres archivos tienen las mismas columnas. unionByName conserva la correspondencia por nombre y permite comprobar que ninguna fila se pierda durante la integración.

In [ ]:
filas_vina = df_vina.count()
filas_santiago = df_santiago.count()
filas_concepcion = df_concepcion.count()

df_ventas_unificado = (
    df_vina
    .unionByName(df_santiago)
    .unionByName(df_concepcion)
)

filas_esperadas = filas_vina + filas_santiago + filas_concepcion
filas_unificadas = df_ventas_unificado.count()

print("Filas Viña del Mar:", filas_vina)
print("Filas Santiago:", filas_santiago)
print("Filas Concepción:", filas_concepcion)
print("Suma de filas originales:", filas_esperadas)
print("Filas del DataFrame unificado:", filas_unificadas)

if filas_unificadas != filas_esperadas:
    raise ValueError("La unión no conserva la suma de filas de las tres fuentes.")

print("Validación de unión: correcta")

# PARTE 2: Calidad inicial antes de limpiar

Se consideran como faltantes tanto los valores nulos como las cadenas vacías. Esto es necesario porque en estos CSV algunas categorías aparecen vacías, aunque no estén representadas como NULL.

In [ ]:
columnas_originales = df_ventas_unificado.columns

def contar_faltantes(df, columnas):
    expresiones = []
    for columna in columnas:
        valor_texto = F.col(columna).cast("string")
        expresiones.append(
            F.sum(
                F.when(
                    F.col(columna).isNull() | (F.trim(valor_texto) == ""),
                    1
                ).otherwise(0)
            ).alias(columna)
        )
    return df.select(expresiones).first().asDict()

filas_antes = df_ventas_unificado.count()
duplicados_antes = filas_antes - df_ventas_unificado.dropDuplicates().count()
nulos_antes = contar_faltantes(df_ventas_unificado, columnas_originales)

cantidad_num_antes = F.col("cantidad").cast("int")
cantidad_invalida_antes = df_ventas_unificado.filter(
    (F.trim(F.col("cantidad")) != "") & cantidad_num_antes.isNull()
).count()
cantidad_no_positiva_antes = df_ventas_unificado.filter(cantidad_num_antes <= 0).count()

print("Filas iniciales:", filas_antes)
print("Duplicados completos:", duplicados_antes)
print("Filas con cantidad no convertible:", cantidad_invalida_antes)
print("Filas con cantidad menor o igual a cero:", cantidad_no_positiva_antes)

tabla_nulos_antes = spark.createDataFrame(
    [(columna, int(nulos_antes[columna])) for columna in columnas_originales],
    ["columna", "faltantes_antes"]
)
tabla_nulos_antes.show(truncate=False)

# PARTE 3: Limpieza y depuración (ID4.2)

Las decisiones se aplican de forma visible:

- Las filas completamente duplicadas se eliminan con dropDuplicates().
- Las categorías vacías se imputan como Sin categoría para conservar esas ventas y diferenciarlas de una categoría real.
- Los nombres de producto se llevan a minúsculas y se eliminan espacios sobrantes.
- Las fechas se convierten usando los formatos observados: dd/MM/yyyy, dd-MM-yyyy, yyyy/MM/dd y yyyy-MM-dd.
- Los precios observados corresponden a pesos enteros, con símbolo $ opcional y espacios. Se retiran esos símbolos antes de convertir a número.

In [ ]:
# 1. Eliminación de duplicados completos.
df_limpio = df_ventas_unificado.dropDuplicates()
print("Duplicados eliminados:", duplicados_antes)
print("Filas después de eliminar duplicados:", df_limpio.count())

# 2. Gestión de categorías vacías o nulas.
df_limpio = df_limpio.withColumn(
    "categoria",
    F.when(
        F.col("categoria").isNull() | (F.trim(F.col("categoria")) == ""),
        F.lit("Sin categoría")
    ).otherwise(F.trim(F.col("categoria")))
)

# 3. Estandarización del nombre del producto.
df_limpio = df_limpio.withColumn(
    "producto_nombre",
    F.lower(F.trim(F.regexp_replace(F.col("producto_nombre"), r"\s+", " ")))
)

# 4. Conversión explícita de fechas.
fecha_original = F.trim(F.col("fecha"))
df_limpio = df_limpio.withColumn("_fecha_original", fecha_original)
df_limpio = df_limpio.withColumn(
    "fecha",
    F.coalesce(
        F.to_date("_fecha_original", "dd/MM/yyyy"),
        F.to_date("_fecha_original", "dd-MM-yyyy"),
        F.to_date("_fecha_original", "yyyy/MM/dd"),
        F.to_date("_fecha_original", "yyyy-MM-dd")
    )
)
fechas_invalidas = df_limpio.filter(
    F.col("_fecha_original").isNotNull() & F.col("fecha").isNull()
).count()

# 5. Limpieza del precio y conversión a número.
precio_original = F.trim(F.col("precio_unitario"))
df_limpio = df_limpio.withColumn("_precio_original", precio_original)
precio_sin_simbolo = F.regexp_replace(F.col("_precio_original"), r"[$\s]", "")
df_limpio = df_limpio.withColumn(
    "precio_unitario",
    precio_sin_simbolo.cast("double")
)
precios_invalidos = df_limpio.filter(
    F.col("_precio_original").isNotNull() & F.col("precio_unitario").isNull()
).count()

# 6. Conversión de cantidad.
cantidad_original = F.trim(F.col("cantidad"))
df_limpio = df_limpio.withColumn("_cantidad_original", cantidad_original)
df_limpio = df_limpio.withColumn("cantidad", F.col("_cantidad_original").cast("int"))
cantidades_invalidas = df_limpio.filter(
    F.col("_cantidad_original").isNotNull() & F.col("cantidad").isNull()
).count()

print("Categorías vacías tratadas como 'Sin categoría'.")
print("Fechas no convertibles:", fechas_invalidas)
print("Precios no convertibles:", precios_invalidos)
print("Cantidades no convertibles:", cantidades_invalidas)

In [ ]:
# Las columnas auxiliares solo se utilizaron para validar conversiones.
df_limpio = df_limpio.drop(
    "_fecha_original", "_precio_original", "_cantidad_original"
)

print("Esquema después de la limpieza:")
df_limpio.printSchema()
df_limpio.show(5, truncate=False)

# PARTE 4: Transformación y validación (ID4.1 e ID4.3)

Se excluyen las filas con cantidad no convertible, cantidad menor o igual a cero, fecha inválida o precio inválido. Las filas excluidas se conservan en un DataFrame separado para poder revisarlas.

In [ ]:
df_rechazadas_cantidad = df_limpio.filter(
    F.col("cantidad").isNull() | (F.col("cantidad") <= 0)
)

df_base_valida = df_limpio.filter(
    F.col("cantidad").isNotNull()
    & (F.col("cantidad") > 0)
    & F.col("precio_unitario").isNotNull()
    & F.col("fecha").isNotNull()
)

df_transformado = (
    df_base_valida
    .withColumn("precio_total", F.col("cantidad") * F.col("precio_unitario"))
    .withColumn("dia_semana", F.date_format(F.col("fecha"), "EEEE"))
)

precio_total_negativo = df_transformado.filter(F.col("precio_total") < 0).count()
df_precio_total_negativo = df_transformado.filter(F.col("precio_total") < 0)

# La regla de calidad exige que el dataset analítico no tenga precio_total negativo.
df_ventas_final = df_transformado.filter(F.col("precio_total") >= 0)

print("Filas rechazadas por cantidad:", df_rechazadas_cantidad.count())
print("precio_total negativos detectados:", precio_total_negativo)
print("Filas finales para el análisis:", df_ventas_final.count())

In [ ]:
print("Cantidad de sucursales únicas:", df_ventas_final.select("sucursal").distinct().count())
df_ventas_final.select("sucursal").distinct().orderBy("sucursal").show(truncate=False)

print("Cantidad de categorías únicas:", df_ventas_final.select("categoria").distinct().count())
df_ventas_final.select("categoria").distinct().orderBy("categoria").show(truncate=False)

print("Ejemplo de columnas calculadas:")
df_ventas_final.select(
    "id_transaccion", "fecha", "dia_semana", "cantidad",
    "precio_unitario", "precio_total"
).show(5, truncate=False)

## Detección de valores atípicos con IQR

Se calcula el rango intercuartílico de precio_unitario. Los registros fuera de Q1 − 1,5 × IQR o Q3 + 1,5 × IQR se marcan con es_outlier_precio. Se conservan en el DataFrame para revisión; no se eliminan automáticamente.

In [ ]:
q1, q3 = df_ventas_final.approxQuantile(
    "precio_unitario", [0.25, 0.75], 0.0
)
iqr = q3 - q1
limite_inferior = q1 - (1.5 * iqr)
limite_superior = q3 + (1.5 * iqr)

df_ventas_final = df_ventas_final.withColumn(
    "es_outlier_precio",
    (F.col("precio_unitario") < F.lit(limite_inferior))
    | (F.col("precio_unitario") > F.lit(limite_superior))
)
df_outliers = df_ventas_final.filter(F.col("es_outlier_precio"))

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Límite inferior:", limite_inferior)
print("Límite superior:", limite_superior)
print("Outliers detectados:", df_outliers.count())
df_outliers.select(
    "id_transaccion", "sucursal", "producto_nombre", "precio_unitario",
    "precio_total", "es_outlier_precio"
).show(20, truncate=False)

## Informe comparativo de calidad

La comparación deja visibles las filas eliminadas, los errores detectados y los outliers marcados. Los outliers se reportan en la columna posterior porque son una señal de revisión, no un error que deba eliminarse automáticamente.

In [ ]:
filas_despues = df_ventas_final.count()
columnas_finales = [
    "id_transaccion", "fecha", "id_producto", "producto_nombre", "categoria",
    "cantidad", "precio_unitario", "id_cliente", "sucursal",
    "precio_total", "dia_semana"
]
nulos_despues = contar_faltantes(df_ventas_final, columnas_finales)

tabla_calidad = spark.createDataFrame([
    ("Filas totales", str(filas_antes), str(filas_despues)),
    ("Duplicados completos", str(duplicados_antes), "0"),
    ("Categorías vacías/nulas", str(nulos_antes["categoria"]), "0"),
    ("Cantidades no convertibles", str(cantidad_invalida_antes), str(cantidades_invalidas)),
    ("Cantidad menor o igual a cero", str(cantidad_no_positiva_antes), "0"),
    ("Fechas no convertibles", str(fechas_invalidas), "0"),
    ("Precios no convertibles", str(precios_invalidos), "0"),
    ("Precio total negativo", "No calculado", str(precio_total_negativo)),
    ("Outliers de precio unitario", "No calculado", str(df_outliers.count())),
], ["indicador", "antes", "despues"])

tabla_calidad.show(truncate=False)

tabla_nulos = spark.createDataFrame(
    [
        (columna, int(nulos_antes.get(columna, 0)), int(nulos_despues.get(columna, 0)))
        for columna in nulos_despues
    ],
    ["columna", "faltantes_antes", "faltantes_despues"]
)
tabla_nulos.show(truncate=False)

# PARTE 5: Análisis con Spark SQL (ID3.3)

Se crea la vista temporal ventas_limpias a partir del DataFrame final. Las tres consultas corresponden a las solicitadas en la actividad.

In [ ]:
df_ventas_final.createOrReplaceTempView("ventas_limpias")
print("Vista temporal creada: ventas_limpias")

## Consulta 1: diez ventas con mayor precio total

Esta consulta muestra el identificador, fecha, sucursal, producto y total de las ventas de mayor monto.

In [ ]:
consulta_1 = spark.sql('''
    SELECT id_transaccion, fecha, sucursal, producto_nombre, precio_total
    FROM ventas_limpias
    ORDER BY precio_total DESC
    LIMIT 10
''')
consulta_1.show(truncate=False)

## Consulta 2: total vendido y transacciones por sucursal

Permite comparar el aporte de cada sucursal considerando el monto total y el número de transacciones válidas.

In [ ]:
consulta_2 = spark.sql('''
    SELECT
        sucursal,
        SUM(precio_total) AS total_ventas,
        COUNT(*) AS cantidad_transacciones
    FROM ventas_limpias
    GROUP BY sucursal
    ORDER BY total_ventas DESC
''')
consulta_2.show(truncate=False)

## Consulta 3: precios por categoría

Se calculan el promedio, mínimo y máximo del precio unitario para cada categoría.

In [ ]:
consulta_3 = spark.sql('''
    SELECT
        categoria,
        AVG(precio_unitario) AS precio_promedio,
        MIN(precio_unitario) AS precio_minimo,
        MAX(precio_unitario) AS precio_maximo
    FROM ventas_limpias
    GROUP BY categoria
    ORDER BY categoria
''')
consulta_3.show(truncate=False)

## Interpretación de resultados

Completa estas observaciones después de ejecutar las tres consultas:

1. En la consulta por sucursal, identifica cuál concentra el mayor total vendido y compara su cantidad de transacciones.
2. En la consulta por categoría, identifica la categoría con mayor precio promedio y revisa si sus valores extremos están asociados a outliers.
3. En la consulta de las diez ventas principales, observa si los montos más altos se explican por una cantidad elevada, por un precio unitario elevado o por ambos.

Las conclusiones deben basarse en las tablas obtenidas durante la ejecución del notebook.

# Cierre y relación con la rúbrica

| Sección | Evidencia | Criterio |
|---|---|---|
| Carga e integración | Tres DataFrames, inspección y comprobación de filas | ID3.1 |
| Limpieza | Nulos de categoría, duplicados, nombres, fechas, precios y cantidades | ID4.2 |
| Transformación | Filtro de cantidades, precio_total, dia_semana y valores únicos | ID4.1 |
| Validación | Regla de precio total, IQR, outliers e informe antes/después | ID4.3 |
| Spark SQL | Vista ventas_limpias y tres consultas solicitadas | ID3.3 |
| Documentación | Explicación Markdown y comentarios en cada decisión | Rúbrica de documentación |

Antes de entregar, ejecuta todas las celdas en Databricks, revisa los resultados y exporta el notebook como mcdi502_s4_grupo7.ipynb. Luego comprímelo en un único archivo ZIP.